l’estrazione dei pesi di attenzione da unmodello Transformer pre-addestrato

In [ ]:
import torch

# Si assume 'model' un encoder Transformer pre-addestrato (es. BERT)
# configurato con output_attentions=True, e 'input_ids' i token
# della sequenza tokenizzata.
model.eval()
with torch.no_grad():
    outputs = model(input_ids, output_attentions=True)

# outputs.attentions: tupla di lunghezza L (numero di layer),
# ciascun elemento di forma (batch, H, n, n)
attentions = outputs.attentions

def attention_rollout(attentions):
    L = len(attentions)
    n = attentions[0].shape[-1]
    rollout = torch.eye(n)
    for layer_attn in attentions:
        # media sugli head: forma (n, n)
        avg_heads = layer_attn[0].mean(dim=0)
        # correzione per la connessione residua
        avg_heads = 0.5 * avg_heads + 0.5 * torch.eye(n)
        rollout = avg_heads @ rollout
    return rollout  # forma (n, n): A_tilde

A_tilde = attention_rollout(attentions)